# Modelo Jerarquico Final de Clones (Type I deterministico + RF para Type II/III/IV)


## Objetivo tecnico
- Mantener `type_I` como regla deterministica (sin RandomForest).
- Entrenar `RandomForest` solo para:
  - Etapa 2: `type_II` vs no-`type_II` dentro de no-`type_I`.
  - Etapa 3: `type_III` vs `type_IV`.
- Aplicar patron didactico (datos -> algoritmo -> evaluacion -> analisis) y patron AST tipo Visitor inspirado en `analisis.py`/`arbol.py`.


In [ ]:
# Librerias base
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import ast
import io
import json
import keyword
import random
import re
import tokenize
import warnings
from difflib import SequenceMatcher

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore", category=SyntaxWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

RUTA_BASE = Path.cwd()
RUTA_DATASET = RUTA_BASE / "DataBaseProject" if (RUTA_BASE / "DataBaseProject").exists() else RUTA_BASE
RUTA_PARES = RUTA_DATASET / "pares_clones"
print("SEED:", SEED)
print("RUTA_DATASET:", RUTA_DATASET)
print("RUTA_PARES:", RUTA_PARES)


In [ ]:
# Configuracion global del experimento
TIPOS_CLON = ["T1", "T2", "T3", "T4"]
TIPO_A_CLASE = {"T1": "type_I", "T2": "type_II", "T3": "type_III", "T4": "type_IV"}
TIPO_A_GRUPO = {"T1": "pares_t1", "T2": "pares_t2", "T3": "pares_t3", "T4": "pares_t4"}

PATRON_SEPARADOR_SNIPPETS = re.compile(r"\n\s*\n\s*\n+")
PATRON_ESPACIOS = re.compile(r"[ \t]+")
PATRON_SALTOS = re.compile(r"\n{3,}")
PATRON_FILE_ID = re.compile(r"_(\d+)\.py$")

UMBRAL_TIPO_I = 1.0
UMBRAL_PROB_TIPO_II = 0.50
MIN_MATCH_LEN_BAKER = 3
AST_VARIANT_OFICIAL = "reduced"
ESTRATEGIA_BALANCEO = "undersample"

ETIQUETAS_MODELO = ["type_I", "type_II", "type_III", "type_IV"]

BAKER_FEATURES_BASE = [
    "baker_match_total_ratio",
    "baker_match_max_ratio",
    "baker_num_blocks",
    "baker_sequence_ratio",
    "baker_edit_distance_norm",
    "baker_token_jaccard",
    "baker_common_token_coverage",
    "baker_len_diff_rel",
]


## 1) Carga del dataset y reconstruccion de pares


In [ ]:
def separar_snippets(texto_archivo: str) -> list[str]:
    texto = texto_archivo.replace("\r\n", "\n").replace("\r", "\n").strip()
    if not texto:
        return []
    return [p.strip() for p in PATRON_SEPARADOR_SNIPPETS.split(texto) if p.strip()]


def cargar_pares_desde_carpetas(ruta_pares: Path) -> pd.DataFrame:
    filas = []
    for tipo in TIPOS_CLON:
        carpeta = ruta_pares / tipo
        if not carpeta.exists():
            raise FileNotFoundError(f"No existe carpeta esperada: {carpeta}")

        for archivo in sorted(carpeta.glob("*.py")):
            contenido = archivo.read_text(encoding="utf-8", errors="replace")
            snippets = separar_snippets(contenido)
            if len(snippets) < 2:
                continue

            m = PATRON_FILE_ID.search(archivo.name)
            if not m:
                continue

            problem_id = int(m.group(1))
            filas.append(
                {
                    "is_clone": 1,
                    "clone_type": TIPO_A_CLASE[tipo],
                    "source_group": TIPO_A_GRUPO[tipo],
                    "filename": archivo.name,
                    "file_path": str(archivo.relative_to(RUTA_DATASET)).replace("/", "\\"),
                    "problem_id": problem_id,
                    "snippet_index_a": 0,
                    "snippet_index_b": 1,
                    "code_a": snippets[0],
                    "code_b": snippets[1],
                }
            )

    if not filas:
        raise RuntimeError("No se pudieron reconstruir pares desde carpetas.")

    return pd.DataFrame(filas)


DatosPares = cargar_pares_desde_carpetas(RUTA_PARES)
print("Filas reconstruidas:", len(DatosPares))
print("Problemas unicos:", DatosPares["problem_id"].nunique())
print("Distribucion por clase:")
print(DatosPares["clone_type"].value_counts().to_string())


## 2) Preprocesamiento lexico para firma Type I y texto base


In [ ]:
def quitar_comentarios(codigo: str) -> str:
    lineas = codigo.expandtabs(4).splitlines()
    limpias = []
    for ln in lineas:
        limpias.append(ln.split("#", 1)[0])
    return "\n".join(limpias)


def normalizar_espacios(codigo: str) -> str:
    lineas = [PATRON_ESPACIOS.sub(" ", l).rstrip() for l in codigo.splitlines()]
    normalizado = "\n".join(lineas).strip()
    return PATRON_SALTOS.sub("\n\n", normalizado)


def preprocesar_codigo(codigo: str) -> str:
    return normalizar_espacios(quitar_comentarios(codigo))


def tokenizar_python_basico(codigo: str) -> list[str]:
    patron = r"[A-Za-z_]\w*|\d+|==|!=|<=|>=|[][(){}.,:;+*/%=<>-]"
    return re.findall(patron, codigo)


def firma_tipo_i_canonica(codigo: str) -> str:
    codigo_ok = preprocesar_codigo(codigo)
    return re.sub(r"\s+", "", codigo_ok)


DatosPares = DatosPares.copy()
DatosPares["code_a_clean"] = DatosPares["code_a"].map(preprocesar_codigo)
DatosPares["code_b_clean"] = DatosPares["code_b"].map(preprocesar_codigo)
DatosPares["tokens_a"] = DatosPares["code_a_clean"].map(tokenizar_python_basico)
DatosPares["tokens_b"] = DatosPares["code_b_clean"].map(tokenizar_python_basico)
DatosPares["token_text_a"] = DatosPares["tokens_a"].map(lambda t: " ".join(t))
DatosPares["token_text_b"] = DatosPares["tokens_b"].map(lambda t: " ".join(t))
DatosPares["type1_signature_a"] = DatosPares["code_a_clean"].map(firma_tipo_i_canonica)
DatosPares["type1_signature_b"] = DatosPares["code_b_clean"].map(firma_tipo_i_canonica)

print("Columnas disponibles:", len(DatosPares.columns))


## 3) Split por `problem_id` y balanceo


In [ ]:
def split_por_grupo(
    df: pd.DataFrame,
    group_col: str,
    target_col: str,
    seed: int = 42,
    train_size: float = 0.7,
    val_size: float = 0.15,
    test_size: float = 0.15,
):
    proporcion_temp = val_size + test_size
    proporcion_test_rel = test_size / proporcion_temp

    gss_train = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=seed)
    idx_train_np, idx_temp_np = next(gss_train.split(df, y=df[target_col], groups=df[group_col]))

    df_temp = df.iloc[idx_temp_np]
    gss_temp = GroupShuffleSplit(n_splits=1, test_size=proporcion_test_rel, random_state=seed)
    idx_val_rel, idx_test_rel = next(gss_temp.split(df_temp, y=df_temp[target_col], groups=df_temp[group_col]))

    return df.index[idx_train_np], df_temp.index[idx_val_rel], df_temp.index[idx_test_rel]


def asignar_split(df: pd.DataFrame, idx_train, idx_val, idx_test, col_split: str = "split") -> pd.DataFrame:
    datos = df.copy()
    datos[col_split] = "unassigned"
    datos.loc[idx_train, col_split] = "train"
    datos.loc[idx_val, col_split] = "val"
    datos.loc[idx_test, col_split] = "test"
    return datos


def estadisticas_split(df: pd.DataFrame, split_col: str, target_col: str, group_col: str) -> list[dict[str, Any]]:
    resumen = []
    for nombre_split, df_split in df.groupby(split_col):
        conteos = df_split[target_col].value_counts().to_dict()
        resumen.append(
            {
                "split": nombre_split,
                "rows": int(len(df_split)),
                "unique_groups": int(df_split[group_col].nunique()),
                "class_distribution": {str(k): int(v) for k, v in conteos.items()},
            }
        )
    return resumen


def balancear_train(df_train: pd.DataFrame, target_col: str, estrategia: str = "none", seed: int = 42):
    conteos = df_train[target_col].value_counts()
    info = {
        "strategy": estrategia,
        "rows_before": int(len(df_train)),
        "class_distribution_before": {str(k): int(v) for k, v in conteos.items()},
    }

    if estrategia == "none" or len(conteos) <= 1:
        info["rows_after"] = int(len(df_train))
        info["class_distribution_after"] = info["class_distribution_before"]
        return df_train.copy(), info

    if estrategia == "undersample":
        n_obj = int(conteos.min())
        rep = False
    elif estrategia == "oversample":
        n_obj = int(conteos.max())
        rep = True
    else:
        info["rows_after"] = int(len(df_train))
        info["class_distribution_after"] = info["class_distribution_before"]
        return df_train.copy(), info

    partes = []
    for clase in conteos.index.tolist():
        df_clase = df_train[df_train[target_col] == clase]
        partes.append(df_clase.sample(n=n_obj, replace=rep, random_state=seed))

    out = pd.concat(partes, axis=0).sample(frac=1.0, random_state=seed).copy()
    c2 = out[target_col].value_counts()
    info["rows_after"] = int(len(out))
    info["class_distribution_after"] = {str(k): int(v) for k, v in c2.items()}
    return out, info


idx_train, idx_val, idx_test = split_por_grupo(
    df=DatosPares,
    group_col="problem_id",
    target_col="clone_type",
    seed=SEED + 100,
    train_size=0.7,
    val_size=0.15,
    test_size=0.15,
)

DatosModelo = asignar_split(DatosPares, idx_train, idx_val, idx_test)

print("Estadisticas split:")
for fila in estadisticas_split(DatosModelo, "split", "clone_type", "problem_id"):
    print(fila)


## 4) Features Baker (base + enriquecidas)


In [ ]:
def _levenshtein_tokens(seq_a: list[str], seq_b: list[str]) -> int:
    n, m = len(seq_a), len(seq_b)
    if n == 0:
        return m
    if m == 0:
        return n

    prev = list(range(m + 1))
    for i in range(1, n + 1):
        curr = [i] + [0] * m
        ai = seq_a[i - 1]
        for j in range(1, m + 1):
            cost = 0 if ai == seq_b[j - 1] else 1
            curr[j] = min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + cost)
        prev = curr
    return int(prev[m])


def _lcs_len_tokens(seq_a: list[str], seq_b: list[str]) -> int:
    if not seq_a or not seq_b:
        return 0
    dp = [0] * (len(seq_b) + 1)
    for xa in seq_a:
        prev = 0
        for j, xb in enumerate(seq_b, start=1):
            temp = dp[j]
            if xa == xb:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = temp
    return int(dp[-1])


def _baker_ngram_set(tokens: list[str], n: int) -> set[tuple[str, ...]]:
    if len(tokens) < n:
        return set()
    return {tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1)}


def _jaccard_set(a: set[Any], b: set[Any]) -> float:
    if not a and not b:
        return 1.0
    union = a | b
    return float(len(a & b) / len(union)) if union else 0.0


def baker_tokenizar_generalizar(codigo: str) -> list[str]:
    codigo_ok = codigo.replace("\r\n", "\n").replace("\r", "\n").expandtabs(4)
    ignorar = {
        tokenize.NL,
        tokenize.NEWLINE,
        tokenize.INDENT,
        tokenize.DEDENT,
        tokenize.COMMENT,
        tokenize.ENCODING,
        tokenize.ENDMARKER,
    }
    out = []
    try:
        flujo = tokenize.generate_tokens(io.StringIO(codigo_ok).readline)
        for tok in flujo:
            if tok.type in ignorar:
                continue
            if tok.type == tokenize.NAME:
                out.append(tok.string if keyword.iskeyword(tok.string) else "ID")
            elif tok.type == tokenize.NUMBER:
                out.append("NUM")
            elif tok.type == tokenize.STRING:
                out.append("STR")
            elif tok.type == tokenize.OP:
                out.append(tok.string)
            else:
                out.append(tok.string)
    except Exception:
        tokens = re.findall(r"[A-Za-z_]\w*|\d+|==|!=|<=|>=|[\[\](){}.,:;+*/%=<>-]", codigo_ok)
        for t in tokens:
            if re.fullmatch(r"[A-Za-z_]\w*", t):
                out.append(t if keyword.iskeyword(t) else "ID")
            elif re.fullmatch(r"\d+", t):
                out.append("NUM")
            else:
                out.append(t)
    return out


def baker_features_par(codigo_a: str, codigo_b: str, min_match_len: int = 3) -> dict[str, float]:
    ta = baker_tokenizar_generalizar(codigo_a)
    tb = baker_tokenizar_generalizar(codigo_b)

    if len(ta) == 0 and len(tb) == 0:
        return {
            "baker_match_total_ratio": 1.0,
            "baker_match_max_ratio": 1.0,
            "baker_num_blocks": 1.0,
            "baker_sequence_ratio": 1.0,
            "baker_edit_distance_norm": 0.0,
            "baker_token_jaccard": 1.0,
            "baker_common_token_coverage": 1.0,
            "baker_len_diff_rel": 0.0,
            "baker_lcs_ratio": 1.0,
            "baker_bigram_jaccard": 1.0,
            "baker_trigram_jaccard": 1.0,
            "baker_keyword_overlap": 1.0,
            "baker_operator_overlap": 1.0,
            "baker_literal_density_diff": 0.0,
            "baker_identifier_density_diff": 0.0,
        }

    if len(ta) == 0 or len(tb) == 0:
        return {
            "baker_match_total_ratio": 0.0,
            "baker_match_max_ratio": 0.0,
            "baker_num_blocks": 0.0,
            "baker_sequence_ratio": 0.0,
            "baker_edit_distance_norm": 1.0,
            "baker_token_jaccard": 0.0,
            "baker_common_token_coverage": 0.0,
            "baker_len_diff_rel": 1.0,
            "baker_lcs_ratio": 0.0,
            "baker_bigram_jaccard": 0.0,
            "baker_trigram_jaccard": 0.0,
            "baker_keyword_overlap": 0.0,
            "baker_operator_overlap": 0.0,
            "baker_literal_density_diff": 1.0,
            "baker_identifier_density_diff": 1.0,
        }

    matcher = SequenceMatcher(a=ta, b=tb, autojunk=False)
    blocks = [b for b in matcher.get_matching_blocks() if b.size >= min_match_len]
    total_match = float(sum(b.size for b in blocks))
    max_match = float(max([b.size for b in blocks], default=0.0))

    base_min = float(min(len(ta), len(tb)))
    base_max = float(max(len(ta), len(tb)))
    set_a, set_b = set(ta), set(tb)
    union = set_a | set_b
    inter = set_a & set_b

    edit_dist = float(_levenshtein_tokens(ta, tb))
    lcs_ratio = float(_lcs_len_tokens(ta, tb) / max(1.0, base_min))
    bigram_j = _jaccard_set(_baker_ngram_set(ta, 2), _baker_ngram_set(tb, 2))
    trigram_j = _jaccard_set(_baker_ngram_set(ta, 3), _baker_ngram_set(tb, 3))

    kw = set(keyword.kwlist)
    kw_a = {t for t in ta if t in kw}
    kw_b = {t for t in tb if t in kw}
    kw_overlap = _jaccard_set(kw_a, kw_b)

    op_pool = {
        "+",
        "-",
        "*",
        "/",
        "//",
        "%",
        "**",
        "=",
        "==",
        "!=",
        "<",
        ">",
        "<=",
        ">=",
        "+=",
        "-=",
        "*=",
        "/=",
        "%=",
        "and",
        "or",
        "not",
        "in",
        "is",
        "(",
        ")",
        "[",
        "]",
        "{",
        "}",
        ".",
        ",",
        ":",
    }
    op_a = {t for t in ta if t in op_pool}
    op_b = {t for t in tb if t in op_pool}
    op_overlap = _jaccard_set(op_a, op_b)

    lit_a = sum(1 for t in ta if t in {"NUM", "STR"}) / max(1.0, len(ta))
    lit_b = sum(1 for t in tb if t in {"NUM", "STR"}) / max(1.0, len(tb))
    id_a = sum(1 for t in ta if t == "ID") / max(1.0, len(ta))
    id_b = sum(1 for t in tb if t == "ID") / max(1.0, len(tb))

    return {
        "baker_match_total_ratio": total_match / base_min,
        "baker_match_max_ratio": max_match / base_min,
        "baker_num_blocks": float(len(blocks)),
        "baker_sequence_ratio": float(matcher.ratio()),
        "baker_edit_distance_norm": edit_dist / max(1.0, base_max),
        "baker_token_jaccard": float(len(inter) / len(union)) if len(union) > 0 else 0.0,
        "baker_common_token_coverage": float(len(inter) / max(1, min(len(set_a), len(set_b)))),
        "baker_len_diff_rel": abs(len(ta) - len(tb)) / max(1.0, base_max),
        "baker_lcs_ratio": lcs_ratio,
        "baker_bigram_jaccard": bigram_j,
        "baker_trigram_jaccard": trigram_j,
        "baker_keyword_overlap": kw_overlap,
        "baker_operator_overlap": op_overlap,
        "baker_literal_density_diff": abs(lit_a - lit_b),
        "baker_identifier_density_diff": abs(id_a - id_b),
    }


def construir_features_baker(df: pd.DataFrame, min_match_len: int = 3) -> pd.DataFrame:
    rows = [baker_features_par(a, b, min_match_len=min_match_len) for a, b in zip(df["code_a_clean"], df["code_b_clean"])]
    return pd.DataFrame(rows, index=df.index)


## 5) AST por patron Visitor (inspirado en `analisis.py` + `arbol.py`)


In [ ]:
def _normalizar_codigo_para_ast(codigo: str) -> str:
    codigo = codigo.replace("\r\n", "\n").replace("\r", "\n").replace("\t", "    ").expandtabs(4)
    lineas = codigo.splitlines()
    out = []
    i = 0
    while i < len(lineas):
        ln = lineas[i]
        s = ln.strip()
        indent_actual = len(ln) - len(ln.lstrip(" "))
        if re.match(r"^(else\s*:|elif\b.*:|except\b.*:|finally\s*:)$", s):
            ln = " " * indent_actual + "if True:"
            s = ln.strip()
        out.append(ln)
        if s and s.endswith(":"):
            j = i + 1
            while j < len(lineas) and lineas[j].strip() == "":
                j += 1
            if j >= len(lineas):
                out.append(" " * (indent_actual + 4) + "pass")
            else:
                indent_sig = len(lineas[j]) - len(lineas[j].lstrip(" "))
                if indent_sig <= indent_actual:
                    out.append(" " * (indent_actual + 4) + "pass")
        i += 1
    texto = "\n".join(out).strip()
    return texto if texto else "pass"


def _diff_rel(a: float, b: float) -> float:
    den = max(1.0, abs(a), abs(b))
    return abs(a - b) / den


def _lcs_len(seq_a: list[str], seq_b: list[str]) -> int:
    if not seq_a or not seq_b:
        return 0
    dp = [0] * (len(seq_b) + 1)
    for xa in seq_a:
        prev = 0
        for j, xb in enumerate(seq_b, start=1):
            temp = dp[j]
            if xa == xb:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = temp
    return int(dp[-1])


def _levenshtein_seq(seq_a: list[str], seq_b: list[str]) -> int:
    n, m = len(seq_a), len(seq_b)
    if n == 0:
        return m
    if m == 0:
        return n
    prev = list(range(m + 1))
    for i in range(1, n + 1):
        curr = [i] + [0] * m
        ai = seq_a[i - 1]
        for j in range(1, m + 1):
            cost = 0 if ai == seq_b[j - 1] else 1
            curr[j] = min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + cost)
        prev = curr
    return int(prev[m])


def _seq_ratio(seq_a: list[str], seq_b: list[str]) -> float:
    return float(SequenceMatcher(a=seq_a, b=seq_b, autojunk=False).ratio())


def _jaccard_keys(mapa_a: dict[str, int], mapa_b: dict[str, int]) -> float:
    ka = set(mapa_a.keys())
    kb = set(mapa_b.keys())
    if not ka and not kb:
        return 1.0
    union = ka | kb
    return float(len(ka & kb) / len(union)) if union else 0.0


def _weighted_overlap_counts(mapa_a: dict[str, int], mapa_b: dict[str, int]) -> float:
    keys = set(mapa_a.keys()) | set(mapa_b.keys())
    if not keys:
        return 1.0
    inter = 0
    total = 0
    for k in keys:
        va = mapa_a.get(k, 0)
        vb = mapa_b.get(k, 0)
        inter += min(va, vb)
        total += max(va, vb)
    return float(inter / total) if total > 0 else 0.0


class VisitorEstructuraAST(ast.NodeVisitor):
    def __init__(self) -> None:
        self.total_nodes = 0
        self.max_depth = 0
        self.num_functions = 0
        self.num_loops = 0
        self.num_ifs = 0
        self.num_calls = 0
        self.num_imports = 0
        self.num_returns = 0
        self.num_assigns = 0
        self.num_comprehensions = 0
        self.num_try = 0
        self.num_branches = 0
        self.num_boolops = 0
        self.num_handlers = 0
        self.ids = set()
        self.type_counts: dict[str, int] = {}

    def visitar(self, root: ast.AST) -> None:
        self._recorrer(root, 0)

    def _recorrer(self, node: ast.AST, depth: int) -> None:
        self.total_nodes += 1
        self.max_depth = max(self.max_depth, depth)
        t = type(node).__name__
        self.type_counts[t] = self.type_counts.get(t, 0) + 1

        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.Lambda)):
            self.num_functions += 1
        if isinstance(node, (ast.For, ast.AsyncFor, ast.While)):
            self.num_loops += 1
        if isinstance(node, (ast.If, ast.IfExp)):
            self.num_ifs += 1
        if isinstance(node, ast.Call):
            self.num_calls += 1
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            self.num_imports += 1
        if isinstance(node, ast.Return):
            self.num_returns += 1
        if isinstance(node, (ast.Assign, ast.AnnAssign, ast.AugAssign)):
            self.num_assigns += 1
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            self.num_comprehensions += 1
        if isinstance(node, (ast.Try, ast.TryStar)):
            self.num_try += 1
        if isinstance(node, (ast.If, ast.For, ast.AsyncFor, ast.While, ast.Try, ast.Match)):
            self.num_branches += 1
        if isinstance(node, ast.BoolOp):
            self.num_boolops += 1
        if isinstance(node, ast.ExceptHandler):
            self.num_handlers += 1
        if isinstance(node, ast.Name):
            self.ids.add(node.id)
        if isinstance(node, ast.arg):
            self.ids.add(node.arg)

        for ch in ast.iter_child_nodes(node):
            self._recorrer(ch, depth + 1)


class VisitorFlujoControl(ast.NodeVisitor):
    def __init__(self) -> None:
        self.flow_sequence: list[str] = []
        self.call_sequence: list[str] = []
        self.branch_count = 0
        self.loop_count = 0
        self.return_count = 0
        self.call_count = 0
        self.try_except_count = 0
        self.max_control_nesting = 0
        self._control_nesting = 0

    def _enter_control(self):
        self._control_nesting += 1
        self.max_control_nesting = max(self.max_control_nesting, self._control_nesting)

    def _exit_control(self):
        self._control_nesting = max(0, self._control_nesting - 1)

    def _call_name(self, func: ast.AST) -> str:
        if isinstance(func, ast.Name):
            return func.id
        if isinstance(func, ast.Attribute):
            base = self._call_name(func.value)
            return f"{base}.{func.attr}" if base else func.attr
        return ""

    def visit_If(self, node: ast.If):
        self.flow_sequence.append("If")
        self.branch_count += 1
        self._enter_control()
        self.generic_visit(node)
        self._exit_control()

    def visit_For(self, node: ast.For):
        self.flow_sequence.append("For")
        self.branch_count += 1
        self.loop_count += 1
        self._enter_control()
        self.generic_visit(node)
        self._exit_control()

    def visit_AsyncFor(self, node: ast.AsyncFor):
        self.visit_For(node)

    def visit_While(self, node: ast.While):
        self.flow_sequence.append("While")
        self.branch_count += 1
        self.loop_count += 1
        self._enter_control()
        self.generic_visit(node)
        self._exit_control()

    def visit_Try(self, node: ast.Try):
        self.flow_sequence.append("Try")
        self.branch_count += 1 + len(node.handlers)
        self.try_except_count += len(node.handlers)
        self._enter_control()
        self.generic_visit(node)
        self._exit_control()

    def visit_TryStar(self, node: ast.TryStar):
        self.visit_Try(node)

    def visit_Return(self, node: ast.Return):
        self.flow_sequence.append("Return")
        self.return_count += 1
        self.generic_visit(node)

    def visit_Call(self, node: ast.Call):
        self.flow_sequence.append("Call")
        self.call_count += 1
        nombre = self._call_name(node.func)
        if nombre:
            self.call_sequence.append(nombre)
        self.generic_visit(node)


class VisitorSemanticaLigera(ast.NodeVisitor):
    def __init__(self) -> None:
        self.op_sequence: list[str] = []
        self.stmt_sequence: list[str] = []
        self.ids = set()

    def visit_BinOp(self, node: ast.BinOp):
        self.op_sequence.append("BIN_" + type(node.op).__name__)
        self.generic_visit(node)

    def visit_BoolOp(self, node: ast.BoolOp):
        self.op_sequence.append("BOOL_" + type(node.op).__name__)
        self.generic_visit(node)

    def visit_UnaryOp(self, node: ast.UnaryOp):
        self.op_sequence.append("UNARY_" + type(node.op).__name__)
        self.generic_visit(node)

    def visit_Compare(self, node: ast.Compare):
        for op in node.ops:
            self.op_sequence.append("CMP_" + type(op).__name__)
        self.generic_visit(node)

    def visit_Name(self, node: ast.Name):
        self.ids.add(node.id)

    def generic_visit(self, node: ast.AST):
        if isinstance(node, ast.stmt):
            self.stmt_sequence.append(type(node).__name__)
        super().generic_visit(node)


## 6) Features AST, IR semantico y complejidad para Type III/IV


In [ ]:
AST_METRIC_KEYS = [
    "ast_total_nodes",
    "ast_depth",
    "ast_num_functions",
    "ast_num_loops",
    "ast_num_ifs",
    "ast_num_calls",
    "ast_num_imports",
    "ast_num_returns",
    "ast_num_assigns",
    "ast_num_comprehensions",
    "ast_num_try",
    "ast_num_branches",
    "ast_cyclomatic",
    "ast_unique_identifiers",
]


def _extraer_fallback_ast(codigo_ok: str) -> dict[str, Any]:
    lineas = codigo_ok.splitlines()
    ids = {x for x in re.findall(r"[A-Za-z_]\w*", codigo_ok) if not keyword.iskeyword(x)}
    ramas = sum(1 for l in lineas if re.match(r"^\s*(if|for|while|try)\b", l))
    return {
        "ast_total_nodes": float(len(tokenizar_python_basico(codigo_ok))),
        "ast_depth": float(max((len(l) - len(l.lstrip(" "))) // 4 for l in lineas if l.strip()) if any(l.strip() for l in lineas) else 0),
        "ast_num_functions": float(sum(1 for l in lineas if re.match(r"^\s*(async\s+def|def)\s+", l))),
        "ast_num_loops": float(sum(1 for l in lineas if re.match(r"^\s*(for|while)\s+", l))),
        "ast_num_ifs": float(sum(1 for l in lineas if re.match(r"^\s*if\s+", l))),
        "ast_num_calls": float(len(re.findall(r"[A-Za-z_]\w*\s*\(", codigo_ok))),
        "ast_num_imports": float(sum(1 for l in lineas if re.match(r"^\s*(import|from)\s+", l))),
        "ast_num_returns": float(sum(1 for l in lineas if re.match(r"^\s*return\b", l))),
        "ast_num_assigns": float(sum(1 for l in lineas if re.search(r"[^=!<>]=[^=]", l))),
        "ast_num_comprehensions": float(len(re.findall(r"\[[^\]]+for\s+.+in\s+.+\]|\{[^\}]+for\s+.+in\s+.+\}", codigo_ok))),
        "ast_num_try": float(sum(1 for l in lineas if re.match(r"^\s*try\s*:", l))),
        "ast_num_branches": float(ramas),
        "ast_cyclomatic": float(1 + ramas),
        "ast_unique_identifiers": float(len(ids)),
        "ast_type_counts": {},
    }


def extraer_features_ast_snippet(codigo: str) -> dict[str, Any]:
    codigo_ok = _normalizar_codigo_para_ast(codigo)
    try:
        root = ast.parse(codigo_ok)
    except SyntaxError:
        return _extraer_fallback_ast(codigo_ok)

    v = VisitorEstructuraAST()
    v.visitar(root)
    cyclomatic = 1 + v.num_branches + v.num_boolops + v.num_handlers
    return {
        "ast_total_nodes": float(v.total_nodes),
        "ast_depth": float(v.max_depth),
        "ast_num_functions": float(v.num_functions),
        "ast_num_loops": float(v.num_loops),
        "ast_num_ifs": float(v.num_ifs),
        "ast_num_calls": float(v.num_calls),
        "ast_num_imports": float(v.num_imports),
        "ast_num_returns": float(v.num_returns),
        "ast_num_assigns": float(v.num_assigns),
        "ast_num_comprehensions": float(v.num_comprehensions),
        "ast_num_try": float(v.num_try),
        "ast_num_branches": float(v.num_branches),
        "ast_cyclomatic": float(cyclomatic),
        "ast_unique_identifiers": float(len(v.ids)),
        "ast_type_counts": v.type_counts,
    }


def extraer_flujo_control_snippet(codigo: str) -> dict[str, Any]:
    codigo_ok = _normalizar_codigo_para_ast(codigo)
    try:
        root = ast.parse(codigo_ok)
    except SyntaxError:
        return {
            "flow_sequence": [],
            "call_sequence": [],
            "branch_count": 0.0,
            "loop_count": 0.0,
            "return_count": 0.0,
            "call_count": 0.0,
            "try_except_count": 0.0,
            "max_control_nesting": 0.0,
        }

    v = VisitorFlujoControl()
    v.visit(root)
    return {
        "flow_sequence": v.flow_sequence,
        "call_sequence": v.call_sequence,
        "branch_count": float(v.branch_count),
        "loop_count": float(v.loop_count),
        "return_count": float(v.return_count),
        "call_count": float(v.call_count),
        "try_except_count": float(v.try_except_count),
        "max_control_nesting": float(v.max_control_nesting),
    }


def extraer_semantica_ligera_snippet(codigo: str) -> dict[str, Any]:
    codigo_ok = _normalizar_codigo_para_ast(codigo)
    try:
        root = ast.parse(codigo_ok)
    except SyntaxError:
        ops = re.findall(r"\+|\-|\*\*|\*|/|//|%|==|!=|<=|>=|<|>", codigo_ok)
        stmts = re.findall(r"^\s*([A-Za-z_]\w*)", codigo_ok, flags=re.M)
        ids = {x for x in re.findall(r"[A-Za-z_]\w*", codigo_ok) if not keyword.iskeyword(x)}
        return {"op_sequence": ops, "stmt_sequence": stmts, "ids": ids}

    v = VisitorSemanticaLigera()
    v.visit(root)
    return {"op_sequence": v.op_sequence, "stmt_sequence": v.stmt_sequence, "ids": v.ids}


# ------------------------------------------------------------
# IR semantico ligero (inspirado en flujo de compilador)
# ------------------------------------------------------------
class VisitorIRSemantico(ast.NodeVisitor):
    def __init__(self) -> None:
        self.ir: list[str] = []
        self.var_map: dict[str, str] = {}
        self.next_var_id = 1

    def _norm_var(self, nombre: str) -> str:
        if nombre not in self.var_map:
            self.var_map[nombre] = f'v{self.next_var_id}'
            self.next_var_id += 1
        return self.var_map[nombre]

    def _expr_token(self, node: ast.AST | None) -> str:
        if node is None:
            return 'NONE'
        if isinstance(node, ast.Name):
            return f'VAR:{self._norm_var(node.id)}'
        if isinstance(node, ast.Constant):
            if isinstance(node.value, bool):
                return 'CONST:BOOL'
            if isinstance(node.value, (int, float, complex)):
                return 'CONST:NUM'
            if isinstance(node.value, str):
                return 'CONST:STR'
            return 'CONST:OTHER'
        if isinstance(node, ast.Call):
            return 'EXPR:CALL'
        if isinstance(node, ast.BinOp):
            return f'EXPR:BIN_{type(node.op).__name__}'
        if isinstance(node, ast.UnaryOp):
            return f'EXPR:UNARY_{type(node.op).__name__}'
        if isinstance(node, ast.BoolOp):
            return f'EXPR:BOOL_{type(node.op).__name__}'
        if isinstance(node, ast.Compare):
            return 'EXPR:COMPARE'
        return f'EXPR:{type(node).__name__}'

    def _call_name(self, func: ast.AST) -> str:
        if isinstance(func, ast.Name):
            return func.id.lower()
        if isinstance(func, ast.Attribute):
            return func.attr.lower()
        return 'call'

    def visit_FunctionDef(self, node: ast.FunctionDef):
        self.ir.append('DEF_FUNC')
        self.generic_visit(node)

    def visit_AsyncFunctionDef(self, node: ast.AsyncFunctionDef):
        self.ir.append('DEF_FUNC')
        self.generic_visit(node)

    def visit_Assign(self, node: ast.Assign):
        token_src = self._expr_token(node.value)
        for t in node.targets:
            if isinstance(t, ast.Name):
                dst = self._norm_var(t.id)
                self.ir.append(f'ASSIGN:{dst}:{token_src}')
            else:
                self.ir.append(f'ASSIGN:TARGET:{token_src}')
        self.generic_visit(node)

    def visit_AugAssign(self, node: ast.AugAssign):
        op = type(node.op).__name__
        lhs = self._expr_token(node.target)
        rhs = self._expr_token(node.value)
        self.ir.append(f'AUGASSIGN:{op}:{lhs}:{rhs}')
        self.generic_visit(node)

    def visit_If(self, node: ast.If):
        self.ir.append('BR_IF')
        self.ir.append('COND:' + self._expr_token(node.test))
        self.generic_visit(node)

    def visit_For(self, node: ast.For):
        self.ir.append('LOOP_FOR')
        self.generic_visit(node)

    def visit_AsyncFor(self, node: ast.AsyncFor):
        self.ir.append('LOOP_FOR')
        self.generic_visit(node)

    def visit_While(self, node: ast.While):
        self.ir.append('LOOP_WHILE')
        self.ir.append('COND:' + self._expr_token(node.test))
        self.generic_visit(node)

    def visit_Try(self, node: ast.Try):
        self.ir.append('TRY_BLOCK')
        self.generic_visit(node)

    def visit_TryStar(self, node: ast.TryStar):
        self.ir.append('TRY_BLOCK')
        self.generic_visit(node)

    def visit_Call(self, node: ast.Call):
        nombre = self._call_name(node.func)
        self.ir.append(f'CALL:{nombre}:{len(node.args)}')
        self.generic_visit(node)

    def visit_Return(self, node: ast.Return):
        self.ir.append('RETURN:' + self._expr_token(node.value))
        self.generic_visit(node)


def extraer_ir_semantico_snippet(codigo: str) -> list[str]:
    codigo_ok = _normalizar_codigo_para_ast(codigo)
    try:
        root = ast.parse(codigo_ok)
    except SyntaxError:
        # fallback muy simple si no parsea
        toks = re.findall(r'if|for|while|return|[A-Za-z_]\w*\s*\(|=', codigo_ok)
        return [f'RAW:{t.strip()}' for t in toks]

    v = VisitorIRSemantico()
    v.visit(root)
    return v.ir


def _ir_features_par(codigo_a: str, codigo_b: str) -> dict[str, float]:
    ir_a = extraer_ir_semantico_snippet(codigo_a)
    ir_b = extraer_ir_semantico_snippet(codigo_b)

    if not ir_a and not ir_b:
        return {
            'ir_sequence_ratio': 1.0,
            'ir_lcs_similarity': 1.0,
            'ir_edit_distance_norm': 0.0,
            'ir_jaccard': 1.0,
            'ir_len_diff_rel': 0.0,
        }

    max_len = max(len(ir_a), len(ir_b))
    min_len = min(len(ir_a), len(ir_b))
    lcs = _lcs_len(ir_a, ir_b)
    edit = _levenshtein_seq(ir_a, ir_b)
    sa, sb = set(ir_a), set(ir_b)
    union = sa | sb
    jacc = 0.0 if not union else float(len(sa & sb) / len(union))

    return {
        'ir_sequence_ratio': _seq_ratio(ir_a, ir_b),
        'ir_lcs_similarity': (1.0 if max_len == 0 else 0.0) if min_len == 0 else float(lcs / min_len),
        'ir_edit_distance_norm': float(edit / max(1, max_len)),
        'ir_jaccard': jacc,
        'ir_len_diff_rel': abs(len(ir_a) - len(ir_b)) / max(1.0, float(max_len)),
    }


COMPLEXITY_METRIC_KEYS = [
    'complexity_cognitive',
    'complexity_halstead_volume',
    'complexity_operators_total',
    'complexity_operators_unique',
    'complexity_operands_total',
    'complexity_max_control_nesting',
    'complexity_writes',
    'complexity_calls',
]


class VisitorComplejidadAST(ast.NodeVisitor):
    # Perfil de complejidad del AST, analogo a una pasada de analisis del compilador.
    def __init__(self) -> None:
        self.operators: list[str] = []
        self.operands: list[str] = []
        self.cognitive = 0
        self.control_nesting = 0
        self.max_control_nesting = 0
        self.writes = 0
        self.calls = 0

    def _visit_control(self, node: ast.AST, operador: str) -> None:
        self.operators.append(operador)
        self.cognitive += 1 + self.control_nesting
        self.control_nesting += 1
        self.max_control_nesting = max(self.max_control_nesting, self.control_nesting)
        self.generic_visit(node)
        self.control_nesting -= 1

    def visit_If(self, node: ast.If):
        self._visit_control(node, 'IF')

    def visit_For(self, node: ast.For):
        self._visit_control(node, 'FOR')

    def visit_AsyncFor(self, node: ast.AsyncFor):
        self._visit_control(node, 'FOR')

    def visit_While(self, node: ast.While):
        self._visit_control(node, 'WHILE')

    def visit_Try(self, node: ast.Try):
        self._visit_control(node, 'TRY')

    def visit_TryStar(self, node: ast.TryStar):
        self._visit_control(node, 'TRY')

    def visit_BinOp(self, node: ast.BinOp):
        self.operators.append('BIN_' + type(node.op).__name__)
        self.generic_visit(node)

    def visit_BoolOp(self, node: ast.BoolOp):
        self.operators.append('BOOL_' + type(node.op).__name__)
        self.cognitive += max(0, len(node.values) - 1)
        self.generic_visit(node)

    def visit_UnaryOp(self, node: ast.UnaryOp):
        self.operators.append('UNARY_' + type(node.op).__name__)
        self.generic_visit(node)

    def visit_Compare(self, node: ast.Compare):
        self.operators.extend('CMP_' + type(op).__name__ for op in node.ops)
        self.generic_visit(node)

    def visit_Assign(self, node: ast.Assign):
        self.operators.append('ASSIGN')
        self.writes += len(node.targets)
        self.generic_visit(node)

    def visit_AugAssign(self, node: ast.AugAssign):
        self.operators.append('AUGASSIGN_' + type(node.op).__name__)
        self.writes += 1
        self.generic_visit(node)

    def visit_Call(self, node: ast.Call):
        self.operators.append('CALL')
        self.calls += 1
        self.generic_visit(node)

    def visit_Name(self, node: ast.Name):
        self.operands.append('ID')

    def visit_Constant(self, node: ast.Constant):
        tipo = 'BOOL' if isinstance(node.value, bool) else type(node.value).__name__.upper()
        self.operands.append('CONST_' + tipo)


def extraer_complejidad_snippet(codigo: str) -> dict[str, float]:
    codigo_ok = _normalizar_codigo_para_ast(codigo)
    try:
        root = ast.parse(codigo_ok)
    except SyntaxError:
        return {k: 0.0 for k in COMPLEXITY_METRIC_KEYS}

    v = VisitorComplejidadAST()
    v.visit(root)
    longitud = len(v.operators) + len(v.operands)
    vocabulario = len(set(v.operators)) + len(set(v.operands))
    volumen = float(longitud * np.log2(max(1, vocabulario)))
    return {
        'complexity_cognitive': float(v.cognitive),
        'complexity_halstead_volume': volumen,
        'complexity_operators_total': float(len(v.operators)),
        'complexity_operators_unique': float(len(set(v.operators))),
        'complexity_operands_total': float(len(v.operands)),
        'complexity_max_control_nesting': float(v.max_control_nesting),
        'complexity_writes': float(v.writes),
        'complexity_calls': float(v.calls),
    }


def construir_features_complejidad_par(df: pd.DataFrame) -> pd.DataFrame:
    fx_a = [extraer_complejidad_snippet(x) for x in df['code_a_clean']]
    fx_b = [extraer_complejidad_snippet(x) for x in df['code_b_clean']]
    filas = []
    for a, b in zip(fx_a, fx_b):
        fila = {}
        for k in COMPLEXITY_METRIC_KEYS:
            fila[f'{k}_diff_abs'] = abs(a[k] - b[k])
            fila[f'{k}_diff_rel'] = _diff_rel(a[k], b[k])
        filas.append(fila)
    return pd.DataFrame(filas, index=df.index)


def construir_features_ast_par(df: pd.DataFrame) -> pd.DataFrame:
    fx_a = [extraer_features_ast_snippet(x) for x in df["code_a_clean"]]
    fx_b = [extraer_features_ast_snippet(x) for x in df["code_b_clean"]]
    filas = []
    for a, b in zip(fx_a, fx_b):
        fila = {}
        for k in AST_METRIC_KEYS:
            va = float(a[k])
            vb = float(b[k])
            fila[f"{k}_diff_abs"] = abs(va - vb)
            fila[f"{k}_diff_rel"] = _diff_rel(va, vb)
        fila["ast_type_jaccard_keys"] = _jaccard_keys(a["ast_type_counts"], b["ast_type_counts"])
        fila["ast_type_weighted_overlap"] = _weighted_overlap_counts(a["ast_type_counts"], b["ast_type_counts"])
        filas.append(fila)
    return pd.DataFrame(filas, index=df.index)


def seleccionar_ast_reducido(df_ast: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "ast_total_nodes_diff_rel",
        "ast_total_nodes_diff_abs",
        "ast_depth_diff_rel",
        "ast_depth_diff_abs",
        "ast_num_functions_diff_rel",
        "ast_num_functions_diff_abs",
        "ast_num_loops_diff_rel",
        "ast_num_loops_diff_abs",
        "ast_num_ifs_diff_rel",
        "ast_num_ifs_diff_abs",
        "ast_num_calls_diff_rel",
        "ast_num_calls_diff_abs",
        "ast_num_returns_diff_rel",
        "ast_num_returns_diff_abs",
        "ast_num_assigns_diff_rel",
        "ast_num_assigns_diff_abs",
        "ast_num_comprehensions_diff_rel",
        "ast_num_try_diff_rel",
        "ast_num_branches_diff_rel",
        "ast_num_branches_diff_abs",
        "ast_cyclomatic_diff_rel",
        "ast_cyclomatic_diff_abs",
        "ast_unique_identifiers_diff_rel",
        "ast_unique_identifiers_diff_abs",
        "ast_type_jaccard_keys",
        "ast_type_weighted_overlap",
    ]
    return df_ast[cols].copy()


def _call_seq_similarity(seq_a: list[str], seq_b: list[str]) -> float:
    if not seq_a and not seq_b:
        return 1.0
    if not seq_a or not seq_b:
        return 0.0
    a = [x.split(".")[-1].lower() for x in seq_a]
    b = [x.split(".")[-1].lower() for x in seq_b]
    return _seq_ratio(a, b)


def construir_features_control_flow_par(df: pd.DataFrame) -> pd.DataFrame:
    fx_a = [extraer_flujo_control_snippet(x) for x in df["code_a_clean"]]
    fx_b = [extraer_flujo_control_snippet(x) for x in df["code_b_clean"]]
    filas = []
    for a, b in zip(fx_a, fx_b):
        seq_a = a["flow_sequence"]
        seq_b = b["flow_sequence"]
        lcs = _lcs_len(seq_a, seq_b)
        min_len = min(len(seq_a), len(seq_b))
        max_len = max(len(seq_a), len(seq_b))
        lcs_similarity = (1.0 if max_len == 0 else 0.0) if min_len == 0 else float(lcs / min_len)
        edit_norm = float(_levenshtein_seq(seq_a, seq_b) / max(1, max_len))
        filas.append(
            {
                "control_flow_sequence_ratio": _seq_ratio(seq_a, seq_b),
                "control_flow_lcs_similarity": lcs_similarity,
                "control_flow_edit_distance": edit_norm,
                "branch_count_diff_rel": _diff_rel(a["branch_count"], b["branch_count"]),
                "loop_count_diff_rel": _diff_rel(a["loop_count"], b["loop_count"]),
                "return_count_diff_rel": _diff_rel(a["return_count"], b["return_count"]),
                "call_count_diff_rel": _diff_rel(a["call_count"], b["call_count"]),
                "try_except_diff_rel": _diff_rel(a["try_except_count"], b["try_except_count"]),
                "max_control_nesting_diff_rel": _diff_rel(a["max_control_nesting"], b["max_control_nesting"]),
                "call_sequence_similarity": _call_seq_similarity(a["call_sequence"], b["call_sequence"]),
            }
        )
    return pd.DataFrame(filas, index=df.index)


def construir_features_ast_enriquecido_par(df: pd.DataFrame) -> pd.DataFrame:
    filas = []
    for a, b in zip(df["code_a_clean"], df["code_b_clean"]):
        sx_a = extraer_semantica_ligera_snippet(a)
        sx_b = extraer_semantica_ligera_snippet(b)

        ops_a, ops_b = sx_a["op_sequence"], sx_b["op_sequence"]
        stmts_a, stmts_b = sx_a["stmt_sequence"], sx_b["stmt_sequence"]
        lcs_ops = _lcs_len(ops_a, ops_b)
        lcs_stmts = _lcs_len(stmts_a, stmts_b)
        min_ops = min(len(ops_a), len(ops_b))
        min_stmts = min(len(stmts_a), len(stmts_b))
        op_lcs = (1.0 if max(len(ops_a), len(ops_b)) == 0 else 0.0) if min_ops == 0 else float(lcs_ops / min_ops)
        stmt_lcs = (1.0 if max(len(stmts_a), len(stmts_b)) == 0 else 0.0) if min_stmts == 0 else float(lcs_stmts / min_stmts)
        ids_a, ids_b = sx_a["ids"], sx_b["ids"]
        union_ids = len(ids_a | ids_b)
        id_jaccard = 1.0 if union_ids == 0 else float(len(ids_a & ids_b) / union_ids)

        fx_ir = _ir_features_par(a, b)

        fila = {
            "ast_operator_sequence_ratio": _seq_ratio(ops_a, ops_b),
            "ast_operator_lcs_similarity": op_lcs,
            "ast_statement_sequence_ratio": _seq_ratio(stmts_a, stmts_b),
            "ast_statement_lcs_similarity": stmt_lcs,
            "ast_identifier_jaccard": id_jaccard,
        }
        fila.update(fx_ir)
        filas.append(fila)
    return pd.DataFrame(filas, index=df.index)


def construir_features_tipo_ii(df: pd.DataFrame, min_match_len: int = 3) -> pd.DataFrame:
    fx = construir_features_baker(df, min_match_len=min_match_len)
    return fx[BAKER_FEATURES_BASE].copy()


def construir_features_tipo_iii_iv(df: pd.DataFrame, min_match_len: int = 3, ast_variant: str = "reduced") -> pd.DataFrame:
    fx_baker = construir_features_baker(df, min_match_len=min_match_len)
    fx_ast = construir_features_ast_par(df)
    fx_ast_sel = seleccionar_ast_reducido(fx_ast) if ast_variant == "reduced" else fx_ast
    fx_ast_extra = construir_features_ast_enriquecido_par(df)
    fx_cf = construir_features_control_flow_par(df)
    fx_complejidad = construir_features_complejidad_par(df)
    return pd.concat([fx_baker, fx_ast_sel, fx_ast_extra, fx_cf, fx_complejidad], axis=1)


## 7) Modelo jerarquico: Type I deterministico + RF2 + RF3


In [ ]:
def es_tipo_i_deterministico(sig_a: str, sig_b: str, umbral: float = 1.0) -> bool:
    if sig_a == sig_b:
        return True
    return SequenceMatcher(None, sig_a, sig_b, autojunk=False).ratio() >= umbral


def detectar_tipo_i_deterministico(df: pd.DataFrame, umbral: float = 1.0) -> pd.Series:
    out = []
    for a, b in zip(df["type1_signature_a"], df["type1_signature_b"]):
        out.append(es_tipo_i_deterministico(a, b, umbral=umbral))
    return pd.Series(out, index=df.index, dtype=bool)


def evaluar_predicciones(y_true, y_pred, labels: list[str]) -> dict[str, Any]:
    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    reporte_dict = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    return {
        "accuracy": float(acc),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "f1_macro": float(f1_macro),
        "confusion_matrix": cm.tolist(),
        "classification_report_dict": reporte_dict,
    }


@dataclass
class ConfigRF:
    n_estimators: int
    max_depth: int | None
    min_samples_leaf: int
    min_samples_split: int
    max_features: str | float | None
    class_weight: str = "balanced_subsample"


def _crear_rf(cfg: ConfigRF, random_state: int) -> RandomForestClassifier:
    return RandomForestClassifier(
        n_estimators=cfg.n_estimators,
        max_depth=cfg.max_depth,
        min_samples_leaf=cfg.min_samples_leaf,
        min_samples_split=cfg.min_samples_split,
        max_features=cfg.max_features,
        class_weight=cfg.class_weight,
        random_state=random_state,
        n_jobs=-1,
    )


def _grilla_rf_tipo2() -> list[ConfigRF]:
    return [
        ConfigRF(500, None, 1, 2, "sqrt"),
        ConfigRF(700, None, 1, 2, "sqrt"),
        ConfigRF(700, 25, 1, 2, "sqrt"),
        ConfigRF(900, None, 2, 5, "sqrt"),
    ]


def _grilla_rf_tipo34() -> list[ConfigRF]:
    return [
        ConfigRF(600, None, 1, 2, "sqrt"),
        ConfigRF(800, None, 1, 2, "sqrt"),
        ConfigRF(900, None, 2, 2, "sqrt"),
        ConfigRF(900, 30, 1, 5, "sqrt"),
    ]


def seleccionar_mejor_rf_tipo2(X_train: pd.DataFrame, y_train: pd.Series, X_val: pd.DataFrame, y_val: pd.Series, seed: int):
    mejor_modelo, mejor_cfg, mejor_f1 = None, None, -1.0
    for cfg in _grilla_rf_tipo2():
        modelo = _crear_rf(cfg, random_state=seed)
        modelo.fit(X_train, y_train)
        pred = modelo.predict(X_val)
        score = f1_score(y_val, pred, average="binary", zero_division=0)
        if score > mejor_f1:
            mejor_modelo, mejor_cfg, mejor_f1 = modelo, cfg, score
    return mejor_modelo, mejor_cfg, float(mejor_f1)


def seleccionar_mejor_rf_tipo34(X_train: pd.DataFrame, y_train: pd.Series, X_val: pd.DataFrame, y_val: pd.Series, seed: int):
    mejor_modelo, mejor_cfg, mejor_f1 = None, None, -1.0
    for cfg in _grilla_rf_tipo34():
        modelo = _crear_rf(cfg, random_state=seed)
        modelo.fit(X_train, y_train)
        pred = modelo.predict(X_val)
        score = f1_score(y_val, pred, average="macro", zero_division=0)
        if score > mejor_f1:
            mejor_modelo, mejor_cfg, mejor_f1 = modelo, cfg, score
    return mejor_modelo, mejor_cfg, float(mejor_f1)


## 8) Entrenamiento, prediccion y evaluacion final


In [ ]:
def predecir_jerarquico(
    df: pd.DataFrame,
    modelo_tipo_ii: RandomForestClassifier,
    modelo_tipo_iii_iv: RandomForestClassifier,
    min_match_len: int = 3,
    ast_variant: str = "reduced",
    umbral_tipo_i: float = 1.0,
    umbral_prob_tipo_ii: float = 0.5,
):
    pred = pd.Series(index=df.index, dtype="object")

    # Capa 1: Type I deterministico
    mask_i = detectar_tipo_i_deterministico(df, umbral=umbral_tipo_i)
    pred.loc[mask_i] = "type_I"

    # Capa 2: RF Type II
    idx_restante = df.index[~mask_i]
    if len(idx_restante) > 0:
        X_t2 = construir_features_tipo_ii(df.loc[idx_restante], min_match_len=min_match_len)
        prob_t2 = modelo_tipo_ii.predict_proba(X_t2)[:, 1]
        pred_t2_bin = (prob_t2 >= umbral_prob_tipo_ii).astype(int)
        idx_type_ii = idx_restante[pred_t2_bin == 1]
        pred.loc[idx_type_ii] = "type_II"

        # Capa 3: RF Type III/IV
        idx_restante_2 = pred.index[pred.isna()]
        if len(idx_restante_2) > 0:
            X_t34 = construir_features_tipo_iii_iv(
                df.loc[idx_restante_2],
                min_match_len=min_match_len,
                ast_variant=ast_variant,
            )
            pred.loc[idx_restante_2] = modelo_tipo_iii_iv.predict(X_t34)

    pred = pred.fillna("type_IV")
    resumen = {
        "pred_type_I": int((pred == "type_I").sum()),
        "pred_type_II": int((pred == "type_II").sum()),
        "pred_type_III": int((pred == "type_III").sum()),
        "pred_type_IV": int((pred == "type_IV").sum()),
    }
    return pred, resumen


def entrenar_evaluar_modelo_jerarquico(
    datos_task: pd.DataFrame,
    columna_target: str,
    etiquetas: list[str],
    seed: int,
    estrategia_balanceo: str = "none",
    min_match_len: int = 3,
    ast_variant: str = "reduced",
    umbral_tipo_i: float = 1.0,
    umbral_prob_tipo_ii: float = 0.5,
):
    train_raw = datos_task[datos_task["split"] == "train"].copy()
    val = datos_task[datos_task["split"] == "val"].copy()
    test = datos_task[datos_task["split"] == "test"].copy()

    train_balanceado, info_balanceo = balancear_train(train_raw, columna_target, estrategia_balanceo, seed)

    mask_i_train = detectar_tipo_i_deterministico(train_balanceado, umbral=umbral_tipo_i)
    train_etapa2 = train_balanceado.loc[~mask_i_train].copy()

    y_train_t2 = (train_etapa2[columna_target] == "type_II").astype(int)
    X_train_t2 = construir_features_tipo_ii(train_etapa2, min_match_len=min_match_len)

    val_no_i_mask = ~detectar_tipo_i_deterministico(val, umbral=umbral_tipo_i)
    val_t2 = val.loc[val_no_i_mask]
    y_val_t2 = (val_t2[columna_target] == "type_II").astype(int)
    X_val_t2 = construir_features_tipo_ii(val_t2, min_match_len=min_match_len)

    modelo_tipo_ii, cfg_t2, f1_t2_val = seleccionar_mejor_rf_tipo2(X_train_t2, y_train_t2, X_val_t2, y_val_t2, seed=seed)

    train_etapa3 = train_etapa2[train_etapa2[columna_target].isin(["type_III", "type_IV"])].copy()
    y_train_t34 = train_etapa3[columna_target]
    X_train_t34 = construir_features_tipo_iii_iv(train_etapa3, min_match_len=min_match_len, ast_variant=ast_variant)

    val_t34 = val_t2[val_t2[columna_target].isin(["type_III", "type_IV"])].copy()
    y_val_t34 = val_t34[columna_target]
    X_val_t34 = construir_features_tipo_iii_iv(val_t34, min_match_len=min_match_len, ast_variant=ast_variant)

    modelo_tipo_iii_iv, cfg_t34, f1_t34_val = seleccionar_mejor_rf_tipo34(X_train_t34, y_train_t34, X_val_t34, y_val_t34, seed=seed + 1030)

    pred_val, resumen_val = predecir_jerarquico(
        val,
        modelo_tipo_ii=modelo_tipo_ii,
        modelo_tipo_iii_iv=modelo_tipo_iii_iv,
        min_match_len=min_match_len,
        ast_variant=ast_variant,
        umbral_tipo_i=umbral_tipo_i,
        umbral_prob_tipo_ii=umbral_prob_tipo_ii,
    )

    pred_test, resumen_test = predecir_jerarquico(
        test,
        modelo_tipo_ii=modelo_tipo_ii,
        modelo_tipo_iii_iv=modelo_tipo_iii_iv,
        min_match_len=min_match_len,
        ast_variant=ast_variant,
        umbral_tipo_i=umbral_tipo_i,
        umbral_prob_tipo_ii=umbral_prob_tipo_ii,
    )

    metricas_val = evaluar_predicciones(val[columna_target], pred_val, labels=etiquetas)
    metricas_test = evaluar_predicciones(test[columna_target], pred_test, labels=etiquetas)

    importancia_t2 = pd.DataFrame({"feature": X_train_t2.columns, "importance": modelo_tipo_ii.feature_importances_}).sort_values("importance", ascending=False)
    importancia_t34 = pd.DataFrame({"feature": X_train_t34.columns, "importance": modelo_tipo_iii_iv.feature_importances_}).sort_values("importance", ascending=False)

    return {
        "info_balanceo": info_balanceo,
        "metricas_val": metricas_val,
        "metricas_test": metricas_test,
        "pred_val": pred_val,
        "pred_test": pred_test,
        "modelo_tipo_ii": modelo_tipo_ii,
        "modelo_tipo_iii_iv": modelo_tipo_iii_iv,
        "num_features_tipo_ii": int(X_train_t2.shape[1]),
        "num_features_tipo_iii_iv": int(X_train_t34.shape[1]),
        "feature_importance_tipo_ii": importancia_t2,
        "feature_importance_tipo_iii_iv": importancia_t34,
        "resumen_pred_val": resumen_val,
        "resumen_pred_test": resumen_test,
        "umbral_tipo_i": float(umbral_tipo_i),
        "umbral_prob_tipo_ii": float(umbral_prob_tipo_ii),
        "cfg_tipo_ii": cfg_t2,
        "cfg_tipo_iii_iv": cfg_t34,
        "f1_val_tipo_ii": f1_t2_val,
        "f1_val_tipo_iii_iv": f1_t34_val,
    }


ResultadoModelo = entrenar_evaluar_modelo_jerarquico(
    datos_task=DatosModelo,
    columna_target="clone_type",
    etiquetas=ETIQUETAS_MODELO,
    seed=SEED + 100,
    estrategia_balanceo=ESTRATEGIA_BALANCEO,
    min_match_len=MIN_MATCH_LEN_BAKER,
    ast_variant=AST_VARIANT_OFICIAL,
    umbral_tipo_i=UMBRAL_TIPO_I,
    umbral_prob_tipo_ii=UMBRAL_PROB_TIPO_II,
)

print("Modelo listo.")
print("Features tipo_II:", ResultadoModelo["num_features_tipo_ii"])
print("Features tipo_III/type_IV:", ResultadoModelo["num_features_tipo_iii_iv"])
print("Mejor F1 val tipo_II:", round(ResultadoModelo["f1_val_tipo_ii"], 6))
print("Mejor F1 val tipo_III/type_IV:", round(ResultadoModelo["f1_val_tipo_iii_iv"], 6))


## 9) Reportes y visualizacion (como en tus notebooks base)


In [ ]:
metricas_val = ResultadoModelo["metricas_val"]
metricas_test = ResultadoModelo["metricas_test"]

print("=== VALIDACION ===")
print("accuracy:", round(metricas_val["accuracy"], 6))
print("f1_macro:", round(metricas_val["f1_macro"], 6))

print("\n=== TEST ===")
print("accuracy:", round(metricas_test["accuracy"], 6))
print("f1_macro:", round(metricas_test["f1_macro"], 6))

reporte_test = pd.DataFrame(metricas_test["classification_report_dict"]).T
display(reporte_test)

cm = np.array(metricas_test["confusion_matrix"])
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=ETIQUETAS_MODELO).plot(cmap="Blues", ax=ax, colorbar=False)
ax.set_title("Matriz de confusion (TEST)")
plt.tight_layout()
plt.show()

print("\nTop 15 features RF tipo_II:")
display(ResultadoModelo["feature_importance_tipo_ii"].head(15))
print("Top 20 features RF tipo_III/type_IV:")
display(ResultadoModelo["feature_importance_tipo_iii_iv"].head(20))


## 10) Resumen del modelo en memoria


In [ ]:
ResumenModelo = {
    "metricas_val": ResultadoModelo["metricas_val"],
    "metricas_test": ResultadoModelo["metricas_test"],
    "resumen_pred_val": ResultadoModelo["resumen_pred_val"],
    "resumen_pred_test": ResultadoModelo["resumen_pred_test"],
    "cfg_tipo_ii": ResultadoModelo["cfg_tipo_ii"].__dict__,
    "cfg_tipo_iii_iv": ResultadoModelo["cfg_tipo_iii_iv"].__dict__,
    "umbral_tipo_i": ResultadoModelo["umbral_tipo_i"],
    "umbral_prob_tipo_ii": ResultadoModelo["umbral_prob_tipo_ii"],
    "num_features_tipo_ii": ResultadoModelo["num_features_tipo_ii"],
    "num_features_tipo_iii_iv": ResultadoModelo["num_features_tipo_iii_iv"],
    "f1_val_tipo_ii": ResultadoModelo["f1_val_tipo_ii"],
    "f1_val_tipo_iii_iv": ResultadoModelo["f1_val_tipo_iii_iv"],
}

print("Resumen del modelo disponible en variable: ResumenModelo")
print("No se escribieron archivos externos.")
print("\nJSON (preview):")
print(json.dumps(ResumenModelo, ensure_ascii=False, indent=2)[:1800] + "\n...")
